# Week 2: seed noise, failure gallery, CLAHE, predict.py
Runtime → Change runtime type → GPU. Run top to bottom.

## 1. Setup
Clones this repo and saves all outputs to Drive (`tooth_seg_week2`).

In [ ]:
!nvidia-smi --query-gpu=name --format=csv
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/tooth_seg_week2'
!mkdir -p $DRIVE

GITHUB_USER = 'varshacvenkat-web'   # change if your repo lives elsewhere
!git clone https://github.com/{GITHUB_USER}/tooth-seg-baseline.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

## 2. Download the dataset
Add your Kaggle token in the 🔑 Secrets panel as `KAGGLE_API_TOKEN` (never paste it into a cell).

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
!pip install -q -U kaggle
!kaggle datasets download -d humansintheloop/teeth-segmentation-on-dental-x-ray-images -p /content/data --unzip -q

## 3. Convert and split (seed 42)
Should print `{'train': 418, 'val': 89, 'test': 91}`, the same split as Week 1.

In [ ]:
!python src/convert.py --src "/content/data/Teeth Segmentation JSON" --out /content/teeth_yolo
!cp /content/teeth_yolo/splits.json $DRIVE/

## 4. Baseline × 3 seeds (~7 min each)
Seed 0 is a rerun of Week 1; it should land near mAP@50:95 0.597, which is itself a reproducibility check. Each run is copied to Drive as soon as it finishes.

In [ ]:
for s in [0, 1, 2]:
    !python src/train.py --data /content/teeth_yolo/data.yaml --seed {s} --name baseline_s{s}
    !cp -r runs/baseline_s{s} $DRIVE/

In [ ]:
!python src/evaluate.py --data /content/teeth_yolo/data.yaml --out results/baseline.csv \
  --runs baseline=runs/baseline_s0/weights/best.pt baseline=runs/baseline_s1/weights/best.pt baseline=runs/baseline_s2/weights/best.pt
!mkdir -p $DRIVE/results && cp results/baseline.csv $DRIVE/results/

## 5. Failure gallery (seed-0 model, 10 worst test images)

In [ ]:
!python src/failures.py --weights runs/baseline_s0/weights/best.pt --data /content/teeth_yolo --out results/failures
!cp -r results/failures $DRIVE/results/
from IPython.display import Image, display
display(Image('results/failures/gallery.jpg', width=1100))

## 6. CLAHE experiment
Point `CLAHE_SRC` at Ansh's copy (same layout as the Kaggle folder: `obj_class_to_machine_color.json` + `d2/img` + `d2/masks_machine`). `--split-file` reuses the baseline's exact train/val/test images. If the copy hasn't arrived, set `USE_ANSH = False` to apply CLAHE ourselves (clip 2.0, 8×8 tiles) and say so in the report.

In [ ]:
USE_ANSH = True
CLAHE_SRC = '/content/drive/MyDrive/clahe_kaggle'   # where Ansh's copy is

if USE_ANSH:
    !python src/convert.py --src "$CLAHE_SRC" --out /content/teeth_yolo_clahe --split-file /content/teeth_yolo/splits.json
else:
    !python src/convert.py --src "/content/data/Teeth Segmentation JSON" --out /content/teeth_yolo_clahe --split-file /content/teeth_yolo/splits.json --clahe

In [ ]:
for s in [0, 1, 2]:
    !python src/train.py --data /content/teeth_yolo_clahe/data.yaml --seed {s} --name clahe_s{s}
    !cp -r runs/clahe_s{s} $DRIVE/

In [ ]:
!python src/evaluate.py --data /content/teeth_yolo_clahe/data.yaml --out results/clahe.csv \
  --runs clahe=runs/clahe_s0/weights/best.pt clahe=runs/clahe_s1/weights/best.pt clahe=runs/clahe_s2/weights/best.pt
!cp results/clahe.csv $DRIVE/results/

## 7. predict.py on a new image
Upload a panoramic X-ray that is not in the dataset (or use a test image).

In [ ]:
from google.colab import files
up = files.upload()
img = list(up)[0]
!python predict.py "$img" --weights runs/baseline_s0/weights/best.pt --out predictions
!cp -r predictions $DRIVE/
from pathlib import Path
display(Image(f'predictions/{Path(img).stem}_overlay.jpg', width=1100))